# Whisper Architecture FLOPs Analysis
This notebook calculates the theoretical Floating Point Operations (FLOPs) required by the **Large** and **Medium** Whisper architectures to process a standard 30-second audio chunk.

Because the compute cost of a Transformer model is dictated by its parameter count and tensor shapes, any model fine-tuned on the same architecture will mathematically share the exact same theoretical FLOPs.

In [ ]:
import torch
from transformers import WhisperForConditionalGeneration
from calflops import calculate_flops

# Define the three transcription scenarios based on word/token count
CASES = {
    "BEST CASE (Silence - 1 token)": 1, # 1 Token: Complete silence. The model instantly outputs a stop token.
    "AVERAGE CASE (Normal Speech - ~100 words / 150 tokens)": 150, # 150 Tokens: An average 30-second chunk containing roughly 100-130 words.
    "WORST CASE (Hallucination - 448 tokens max)": 448 # 448 Tokens: The absolute maximum capacity for a Whisper chunk (often seen during hallucination loops).
}

# Whisper strictly processes audio at 100 frames per second.
# Therefore, 30 seconds of audio is always exactly 3000 frames.
AUDIO_FRAMES_30_SEC = 3000

### 1. Large Architecture (1.55 Billion Parameters)
The **Whisper Large** architecture (including Large-v1, v2, and v3) contains roughly 1.55 Billion parameters.  
**Note:** Whether you use the standard OpenAI baseline or a fine-tuned version like `mozilla-ai/whisper-large-v3-bn`, the theoretical FLOPs will be absolutely identical. We will use the Mozilla-AI weights here to represent the entire Large architecture family.

In [ ]:
print("Loading Large Architecture...")
large_model_id = "mozilla-ai/whisper-large-v3-bn"
model_large = WhisperForConditionalGeneration.from_pretrained(large_model_id)

mel_bins_large = getattr(model_large.config, "num_mel_bins", 128)

audio_tensor_large = torch.randn(1, mel_bins_large, AUDIO_FRAMES_30_SEC)

print(f"\n--- Profiling Large Architecture ({large_model_id}) ---")
for case_name, num_tokens in CASES.items():
    print(f"\n> {case_name}")
    
    text_tokens = torch.randint(0, model_large.config.vocab_size, (1, num_tokens))
    
    kwargs = {
        "input_features": audio_tensor_large,
        "decoder_input_ids": text_tokens
    }
    
    flops, macs, params = calculate_flops(
        model=model_large,
        kwargs=kwargs,
        print_results=False
    )
    
    print(f"Parameters: {params}")
    print(f"MACs:       {macs}")
    print(f"FLOPs:      {flops}")

Loading Large Architecture...


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]


--- Profiling Large Architecture (mozilla-ai/whisper-large-v3-bn) ---

> BEST CASE (Silence - 1 token)
Parameters: 1.54 B
MACs:       1.11 TMACs
FLOPs:      2.22 TFLOPS

> AVERAGE CASE (Normal Speech - ~100 words / 150 tokens)
Parameters: 1.54 B
MACs:       1.23 TMACs
FLOPs:      2.46 TFLOPS

> WORST CASE (Hallucination - 448 tokens max)
Parameters: 1.54 B
MACs:       1.47 TMACs
FLOPs:      2.94 TFLOPS


### 2. Medium Architecture (769 Million Parameters)
The **Whisper Medium** architecture contains roughly 769 Million parameters—about half the size of the Large models.  
**Note:** Just like above, any model fine-tuned on this architecture (such as the BengaliAI competition winner `tugstugi` or DL Sprint 4.0's `bitwisemind-sam`) will share the exact same theoretical FLOPs. We will load the local `bitwisemind_sam` source folder here to represent the entire Medium architecture family.

In [ ]:
print("Loading Medium Architecture...")
medium_model_id = "models/bitwisemind_sam"
model_medium = WhisperForConditionalGeneration.from_pretrained(medium_model_id)

mel_bins_medium = getattr(model_medium.config, "num_mel_bins", 80)

audio_tensor_medium = torch.randn(1, mel_bins_medium, AUDIO_FRAMES_30_SEC)

print(f"\n--- Profiling Medium Architecture ({medium_model_id}) ---")
for case_name, num_tokens in CASES.items():
    print(f"\n> {case_name}")
    
    text_tokens = torch.randint(0, model_medium.config.vocab_size, (1, num_tokens))
    
    kwargs = {
        "input_features": audio_tensor_medium,
        "decoder_input_ids": text_tokens
    }
    
    flops, macs, params = calculate_flops(
        model=model_medium,
        kwargs=kwargs,
        print_results=False
    )
    
    print(f"Parameters: {params}")
    print(f"MACs:       {macs}")
    print(f"FLOPs:      {flops}")

Loading Medium Architecture...


Loading weights:   0%|          | 0/947 [00:00<?, ?it/s]


--- Profiling Medium Architecture (models/bitwisemind_sam) ---

> BEST CASE (Silence - 1 token)
Parameters: 763.86 M
MACs:       534.34 GMACs
FLOPs:      1.07 TFLOPS

> AVERAGE CASE (Normal Speech - ~100 words / 150 tokens)
Parameters: 763.86 M
MACs:       594.75 GMACs
FLOPs:      1.19 TFLOPS

> WORST CASE (Hallucination - 448 tokens max)
Parameters: 763.86 M
MACs:       715.57 GMACs
FLOPs:      1.43 TFLOPS
